In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from typing import  List
from langchain.tools import tool
from langchain_community.tools import TavilySearchResults


@tool
def ask_user_input(
        question: str,
        options: List[str],
        allow_freetext:bool=True,
        user_answer:str=""
)->str:
    """
    Ask the user for input needed to complete the task.
    Call once per missing piece of information — ONE question at a time.
    Generate options SPECIFIC to the user's exact request context.
    Leave user_answer empty — the human fills it in via the review UI.
    Returns the human's answer as a string.
    """
    return user_answer

tavily_search=TavilySearchResults(
    max_results=5,
    description=(
        "Search the web for current, real-time, or external information. "
        "Use when the user asks about recent events, live prices, current news, "
        "latest research, real-time data, or anything that may have changed "
        "since your training cutoff. Do NOT use for general knowledge you already know."

    )
)
ALL_TOOLS=[
    ask_user_input,
    tavily_search,
]
print("Tools:", [t.name for t in ALL_TOOLS])
print("interrupt_on will watch: ask_user_input only")
print("tavily_search auto-executes — no pause, no approval")

Tools: ['ask_user_input', 'tavily_search_results_json']
interrupt_on will watch: ask_user_input only
tavily_search auto-executes — no pause, no approval


In [4]:
from langchain.agents import create_agent,AgentState
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
SYSTEM_PROMPT="""You are a helpful assistant. You can help with anything the user asks.

You have two tools:
  ask_user_input  — ask the user when YOU need their input to proceed
  tavily_search   — search the web when YOU need current or external information

## When to call ask_user_input

Call it when the request is ambiguous and only the USER can resolve it:
  - A specific preference   (which city, which tone, which format)
  - A key unknown detail    (recipient name, travel date, target audience)
  - An ambiguous choice     (multiple valid interpretations)

Do NOT call it when:
  - The request has enough context to answer well  → respond directly
  - You need external data                         → use tavily_search instead
  - A reasonable default clearly applies           → use it and respond

## When to call tavily_search

Call it when you need information that may have changed since your training
or that requires real-time / external data:
  - Current news, recent events, today's prices
  - Latest research papers, recent drug approvals
  - Live flight schedules, current availability
  - Real-time weather, stock prices, sports scores
  - Any fact the user would expect to be up-to-date

Do NOT call it for general knowledge you already know well.

## How to generate options for ask_user_input

Options must be SPECIFIC to the user's exact request — never generic:
  User: 'flights from India to China'
    question: 'Which city in India are you flying from?'
    options:  ['Delhi (DEL)', 'Mumbai (BOM)', 'Hyderabad (HYD)', 'Chennai (MAA)', 'Other']

  User: 'follow-up email'
    question: 'What tone should the email have?'
    options:  ['Professional and polite', 'Apologetic', 'Firm / urgent', 'Friendly']

  NEVER use: ['Option A', 'Option B'] — always derive from context.

## After collecting inputs or searching

Respond completely and directly. Give the user something fully usable.
"""
agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=ALL_TOOLS,
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "ask_user_input": True,
        },
    ),
    ],
    system_prompt=SYSTEM_PROMPT,
)
print("agent is ready")
print("tools ask_user_input,tavily_search")
print("interrupt_on : {ask_user_input:True}-tavily_search auto executes,no pause")

agent is ready
tools ask_user_input,tavily_search
interrupt_on : {ask_user_input:True}-tavily_search auto executes,no pause


In [5]:
from langchain.messages import HumanMessage
from langgraph.types import Command


def handle_ask_user_input(action_request: dict) -> Command:
    """
    Display the LLM-generated question + options.
    Resume with 'edit' to inject the human's answer as user_answer.
    Identical resume format to notebook 3.3.
    """
    args           = action_request["args"]
    question       = args["question"]
    options        = args["options"]
    allow_freetext = args.get("allow_freetext", True)

    print("\n" + "─" * 56)
    print(f"  {question}\n")
    for i, opt in enumerate(options, 1):
        print(f"  {i}. {opt}")
    if allow_freetext:
        print("  (or type your own answer)")
    print("─" * 56)

    raw = input("  Your answer (number or text): ").strip()

    # Resolve number → option text
    if raw.isdigit() and 1 <= int(raw) <= len(options):
        user_answer = options[int(raw) - 1]
    else:
        user_answer = raw

    print(f"  ✓ {user_answer}")

    # Resume with edit — inject user_answer.
    # ask_user_input runs and returns user_answer as a ToolMessage to the LLM.
    return Command(
        resume={
            "decisions": [{
                "type": "edit",
                "edited_action": {
                    "name": "ask_user_input",
                    "args": {
                        "question":       question,
                        "options":        options,
                        "allow_freetext": allow_freetext,
                        "user_answer":    user_answer,   # ← injected
                    }
                }
            }]
        }
    )


def run_claude_style_agent(user_message: str, thread_id: str):
    """
    Run the Claude-style agent with HITL.

    Loop:
      __interrupt__ present  → LLM needs input → show Q&A → resume
      __interrupt__ absent   → LLM has everything → final response
    """
    config = {"configurable": {"thread_id": thread_id}}
    current_input = {"messages": [HumanMessage(content=user_message)]}

    print(f"\n{'='*56}")
    print(f"  You: {user_message}")
    print(f"{'='*56}")

    while True:
        response = agent.invoke(current_input, config=config)

        if not response.get("__interrupt__"):
            # LLM has all inputs → responded directly from its knowledge
            final = response["messages"][-1].content
            print(f"\n  Agent:\n")
            print(final)
            print(f"\n{'─'*56}\n")
            return response

        # Only one interrupt type: ask_user_input
        action_request = response["__interrupt__"][0].value["action_requests"][0]
        current_input  = handle_ask_user_input(action_request)


print("Loop ready.")

Loop ready.


In [7]:
run_claude_style_agent(
    user_message="Iam traveling from Hyd to Bangalore can you find the flights for me?",
    thread_id='travel-02'
)


  You: Iam traveling from Hyd to Bangalore can you find the flights for me?

  Agent:

There are several flights available from Hyderabad to Bangalore. Here are some options:

* IndiGo: 64 direct flights per week, with prices starting from $80
* Air India Express: 40 direct flights per week, with prices starting from $79
* Alliance Air: 2 direct flights per week, with prices starting from $47
* Akasa Air: 1 direct flight per week, with prices starting from $34
* Air India: 219 connecting flights per week, with prices starting from $70

You can book your flight by visiting the websites of these airlines or by using a flight booking platform like Google Flights, KAYAK, or FareEagle. It's always a good idea to compare prices and book in advance to get the best deal.

────────────────────────────────────────────────────────



{'messages': [HumanMessage(content='Iam traveling from india to china can you find the flights for me?', additional_kwargs={}, response_metadata={}, id='71c46fad-ab95-45b2-8b5e-a4de3d06b35d'),
  HumanMessage(content='Iam traveling from Hyd to Bangalore can you find the flights for me?', additional_kwargs={}, response_metadata={}, id='8e60d0a9-dafd-4be3-ab37-66be7826e147'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2qkgt7zry', 'function': {'arguments': '{"query":"flights from Hyderabad to Bangalore"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 998, 'total_tokens': 1021, 'completion_time': 0.063623936, 'completion_tokens_details': None, 'prompt_time': 0.387108108, 'prompt_tokens_details': None, 'queue_time': 0.167163096, 'total_time': 0.450732044}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reaso

In [8]:
run_claude_style_agent(
    user_message="send a follow-up email to my client about the delayed project delivery",
    thread_id="email-01"
)


  You: send a follow-up email to my client about the delayed project delivery

────────────────────────────────────────────────────────
  What tone should the follow-up email have?

  1. Professional and polite
  2. Apologetic
  3. Firm / urgent
  4. Friendly
  (or type your own answer)
────────────────────────────────────────────────────────
  ✓ Firm / urgent

────────────────────────────────────────────────────────
  What is the client's name?

  (or type your own answer)
────────────────────────────────────────────────────────
  ✓ Nehish

  Agent:

Here's a follow-up email to your client, Nehish, about the delayed project delivery:

Subject: Update on Project Delivery

Dear Nehish,

I am writing to provide you with an update on the project that was due for delivery on [Original Due Date]. Unfortunately, due to unforeseen circumstances, we have encountered some delays and the project will not be completed on time.

I want to assure you that we are working diligently to complete the 

{'messages': [HumanMessage(content='send a follow-up email to my client about the delayed project delivery', additional_kwargs={}, response_metadata={}, id='3ba3cfd7-337b-4ac3-9e7d-2c2625416180'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'p4dre0yvk', 'function': {'arguments': '{"allow_freetext":true,"options":["Professional and polite","Apologetic","Firm / urgent","Friendly"],"question":"What tone should the follow-up email have?","user_answer":""}', 'name': 'ask_user_input'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 975, 'total_tokens': 1034, 'completion_time': 0.187636221, 'completion_tokens_details': None, 'prompt_time': 0.077686867, 'prompt_tokens_details': None, 'queue_time': 0.16157443, 'total_time': 0.265323088}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id